###### DSM500 - MSc Thesis: Investigating the Performance and Reliability of Deep-Learning Flood Segmentation Under Varying Flood Extent Using Sentinel-1 SAR Imagery
###### 29 July 2026
###### Archie Hulse
###### Student Number: 250410903
##### Notebook 2: 
#                                               02 - <u>Data Pre-processing<u/>
--------------------------------


#### <u>Contents:</u>
###### *(Hyperlinks to sections of this report)*

[1.1](#1.1) Import libraries, define paths and Clean

[2.1](#2.1) Load Train, Validation and Test Splits

[3.1](#3.1) Add Split Information

[4.1](#4.1) Verify Satellite Image - Label Pairs

[5.1](#5.1) Extract Flood Event

[6.1](#6.1) Assess Image Dimensions and Bands

[7.1](#7.1) Calculate flood extent for every chip

[8.1](#8.1) Add Image Dimensions

[9.1](#9.1) Final Data Table

[9.2](#9.2) Bolivia Holdout Table

[10.1](#10.1)Final Checks & Save

[Go to References](#R) - References

------------------------------

###### 1.1
### Section 1 — Import libraries and define paths:

In [1]:
# Core Libraries:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

In [2]:
# Define the project root directory.
# The notebook is stored inside the "Notebooks" folder,
# so the project directory is one level above the notebook.

PROJECT_DIR = Path.cwd().parent

# Main project directories

DATA_DIR = PROJECT_DIR / "Data"
RESULTS_DIR = PROJECT_DIR / "Results"

# Sen1Floods11 dataset

SEN1_DIR = DATA_DIR / "Sen1Floods11_raw" / "v1.1"

# Image and label directories

S1_DIR = (
    SEN1_DIR
    / "data"
    / "flood_events"
    / "HandLabeled"
    / "S1Hand"
)

LABEL_DIR = (
    SEN1_DIR
    / "data"
    / "flood_events"
    / "HandLabeled"
    / "LabelHand"
)

# Official Sen1Floods11 train/validation/test splits.
#
# The dataset and its evaluation framework were introduced by:
#
# Bonafilia et al. (2020), "Sen1Floods11: A Georeferenced Dataset
# to Train and Test Deep Learning Flood Algorithms for Sentinel-1".
#
# DOI: 10.1109/CVPRW50498.2020.00113

SPLIT_DIR = (
    SEN1_DIR
    / "splits"
    / "flood_handlabeled"
)

# Output directories

TABLE_DIR = RESULTS_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("S1 directory:", S1_DIR)
print("Label directory:", LABEL_DIR)
print("Split directory:", SPLIT_DIR)
print("Output directory:", TABLE_DIR)

Project directory: /Users/archiewilliamhulse/Desktop/Msc Data Science/Final Project/DSM500 Final Project
S1 directory: /Users/archiewilliamhulse/Desktop/Msc Data Science/Final Project/DSM500 Final Project/Data/Sen1Floods11_raw/v1.1/data/flood_events/HandLabeled/S1Hand
Label directory: /Users/archiewilliamhulse/Desktop/Msc Data Science/Final Project/DSM500 Final Project/Data/Sen1Floods11_raw/v1.1/data/flood_events/HandLabeled/LabelHand
Split directory: /Users/archiewilliamhulse/Desktop/Msc Data Science/Final Project/DSM500 Final Project/Data/Sen1Floods11_raw/v1.1/splits/flood_handlabeled
Output directory: /Users/archiewilliamhulse/Desktop/Msc Data Science/Final Project/DSM500 Final Project/Results/tables


In [3]:
# Identify chips with 100% ignore/no-data labels.
#
# These chips contain no valid ground-truth pixels, so a meaningful
# flood extent cannot be calculated and they cannot be used for training
# or evaluation. They are excluded from the experimental dataset but
# remain untouched in the original raw dataset.

invalid_label_ids = []

for file in LABEL_DIR.glob("*_LabelHand.tif"):

    with rasterio.open(file) as src:
        mask = src.read(1)

    # -1 represents the ignore/no-data class.
    if np.all(mask == -1):
        invalid_label_ids.append(
            file.name.replace("_LabelHand.tif", "")
        )

raw_hand_labelled = len(list(LABEL_DIR.glob("*_LabelHand.tif")))

print("Raw hand-labelled chips:", raw_hand_labelled)
print("Chips with 100% ignore pixels:", len(invalid_label_ids))
print("Excluded chips:", invalid_label_ids)
print(
    "Usable experimental chips:",
    raw_hand_labelled - len(invalid_label_ids)
)

Raw hand-labelled chips: 446
Chips with 100% ignore pixels: 5
Excluded chips: ['Ghana_83483', 'Ghana_26376', 'Ghana_5079', 'Ghana_234935', 'Ghana_277']
Usable experimental chips: 441


---------------------------------
###### 2.1
### Section 2 — Load the Official train/validation/test Splits:

In [4]:
# Load the official Sen1Floods11 train/validation/test splits:

train = pd.read_csv(
    SPLIT_DIR / "flood_train_data.csv",
    header=None,
    names=["s1_filename", "label_filename"])

valid = pd.read_csv(
    SPLIT_DIR / "flood_valid_data.csv",
    header=None,
    names=["s1_filename", "label_filename"])

test = pd.read_csv(
    SPLIT_DIR / "flood_test_data.csv",
    header=None,
    names=["s1_filename", "label_filename"])


# Identify the five chips with 100% ignore/no-data labels.
invalid_label_ids = []
for file in LABEL_DIR.glob("*_LabelHand.tif"):
    with rasterio.open(file) as src:
        mask = src.read(1)
    # -1 represents the ignore/no-data class.
    if np.all(mask == -1):
        invalid_label_ids.append(
            file.name.replace("_LabelHand.tif", ""))


# Remove these invalid chips from the official splits.
for split in [train, valid, test]:
    split["s1_id"] = (
        split["s1_filename"]
        .str.replace("_S1Hand.tif", "", regex=False))

    split.drop(
        split[split["s1_id"].isin(invalid_label_ids)].index,
        inplace=True)
    split.drop(columns="s1_id", inplace=True)


# Resulting experimental dataset:
print("Raw hand-labelled chips: 446")
print("Excluded chips:", len(invalid_label_ids))
print("Usable experimental chips:",len(train) + len(valid) + len(test))

print("\nTrain:", len(train))
print("Validation:", len(valid))
print("Test:", len(test))
print("Total:", len(train) + len(valid) + len(test))

Raw hand-labelled chips: 446
Excluded chips: 5
Usable experimental chips: 426

Train: 251
Validation: 86
Test: 89
Total: 426


---------------------------
###### 3.1
### Section 3 — Add Split Information:

In [5]:
# Combine into a single dataframe:

train["split"] = "train"
valid["split"] = "validation"
test["split"] = "test"

splits = pd.concat([train, valid, test], ignore_index=True)

print(splits.sample(10))
print()
print(splits["split"].value_counts())

                     s1_filename                  label_filename       split
392     Somalia_60129_S1Hand.tif     Somalia_60129_LabelHand.tif        test
178     Spain_1167260_S1Hand.tif     Spain_1167260_LabelHand.tif       train
264      India_244057_S1Hand.tif      India_244057_LabelHand.tif  validation
289   Paraguay_305760_S1Hand.tif   Paraguay_305760_LabelHand.tif  validation
256      Ghana_124834_S1Hand.tif      Ghana_124834_LabelHand.tif  validation
206  Sri-Lanka_956740_S1Hand.tif  Sri-Lanka_956740_LabelHand.tif       train
221        USA_375183_S1Hand.tif        USA_375183_LabelHand.tif       train
298     Paraguay_7894_S1Hand.tif     Paraguay_7894_LabelHand.tif  validation
26       Ghana_194723_S1Hand.tif      Ghana_194723_LabelHand.tif       train
147   Paraguay_224845_S1Hand.tif   Paraguay_224845_LabelHand.tif       train

split
train         251
test           89
validation     86
Name: count, dtype: int64


-------------------------------
###### 4.1
### Section 4 — Verify Satellite Image - Label Pairs:

In [6]:
# Quality and Error Control: (check to see if every Satellite Image has a matching flood label)

splits["s1_path"] = splits["s1_filename"].apply(lambda x: S1_DIR / x)
splits["label_path"] = splits["label_filename"].apply(lambda x: LABEL_DIR / x)
splits["s1_exists"] = splits["s1_path"].apply(Path.exists)
splits["label_exists"] = splits["label_path"].apply(Path.exists)

print("Missing SAR images:", (~splits["s1_exists"]).sum())
print("Missing labels:", (~splits["label_exists"]).sum())

Missing SAR images: 0
Missing labels: 0


In [7]:
# Confirm that every image and label referenced by the official
# dataset splits actually exists before continuing.
# The assertions stop execution if any file is missing.

assert splits["s1_exists"].all()
assert splits["label_exists"].all()

print("✓ Every SAR image has a corresponding label.")

✓ Every SAR image has a corresponding label.


In [8]:
# Also check filenames actually correspond:

splits["s1_id"] = (
    splits["s1_filename"]
    .str.replace("_S1Hand.tif", "", regex=False))

splits["label_id"] = (
    splits["label_filename"]
    .str.replace("_LabelHand.tif", "", regex=False))

print("Mismatched pairs:", (splits["s1_id"] != splits["label_id"]).sum())

Mismatched pairs: 0


In [9]:
assert (splits["s1_id"] == splits["label_id"]).all()

print("✓ All SAR images and labels are correctly matched.")

✓ All SAR images and labels are correctly matched.


----------------------------
###### 5.1
### Section 5 — Extract Flood Event:

In [10]:
# Use the File name to extract the Flood event:

splits["flood_event"] = (splits["s1_filename"].str.split("_").str[0])

print(splits["flood_event"].value_counts())

flood_event
USA          69
India        68
Paraguay     67
Ghana        48
Sri-Lanka    42
Mekong       30
Spain        30
Pakistan     28
Somalia      26
Nigeria      18
Name: count, dtype: int64


-----------------------------
###### 6.1
### Section 6 — Assess Image Dimensions and Bands:

In [11]:
# Function to open and inspect SAR Satellite Images:

def inspect_sar_image(path):
    with rasterio.open(path) as src:
        return {
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "crs": str(src.crs),
            "dtype": src.dtypes[0]
        }

In [12]:
# Test 'inspect_sar_image' function:

example_info = inspect_sar_image(splits.iloc[0]["s1_path"])
example_info

{'width': 512,
 'height': 512,
 'bands': 2,
 'crs': 'EPSG:4326',
 'dtype': 'float32'}

---------------------------
###### 7.1
### Section 7 — Calculate flood extent for every chip:

Every Pixel has a label of either:

    -1 = ignore/no data
    0 = background
    1 = flood

In [13]:
# Function to calculate Flood Extent(%) Variable:

def calculate_flood_extent(label_path):
    """
    Calculate the percentage of valid pixels classified as flooded.

    The Sen1Floods11 hand-labelled masks use:
        -1 = ignore / invalid pixel
         0 = background
         1 = flood

    Pixels labelled -1 are excluded from the calculation.

    Parameters
    ----------
    label_path : pathlib.Path or str
        Path to a Sen1Floods11 flood label raster.

    Returns
    -------
    dict
        Dictionary containing:
        - valid_pixels: number of non-ignored pixels
        - flood_pixels: number of pixels labelled as flood
        - flood_extent_percent: percentage of valid pixels classified
          as flooded
    """

    with rasterio.open(label_path) as src:
        mask = src.read(1)

    # Exclude pixels labelled -1 as invalid/ignored.
    valid = mask != -1

    valid_pixels = np.sum(valid)
    flood_pixels = np.sum(mask == 1)

    # Handle images containing no valid pixels.
    if valid_pixels == 0:
        return {
            "valid_pixels": 0,
            "flood_pixels": 0,
            "flood_extent_percent": np.nan
        }

    # Calculate flood extent as a percentage of valid pixels.
    flood_extent = (flood_pixels / valid_pixels) * 100

    return {
        "valid_pixels": valid_pixels,
        "flood_pixels": flood_pixels,
        "flood_extent_percent": flood_extent
    }

In [14]:
# Apply the 'calculate_flood_extent' function to every chip:

extent_results = []

for _, row in splits.iterrows():
    result = calculate_flood_extent(row["label_path"])
    extent_results.append(result)
extent_df = pd.DataFrame(extent_results)

splits = pd.concat(
    [splits.reset_index(drop=True),
     extent_df.reset_index(drop=True)],
    axis=1)

In [15]:
splits[["s1_filename", "split", "flood_event", "valid_pixels", "flood_pixels", "flood_extent_percent"]].sample(10)

,s1_filename,split,flood_event,valid_pixels,flood_pixels,flood_extent_percent
423,USA_670826_S1Hand.tif,test,USA,262132,3010,1.148276
51,India_981708_S1Hand.tif,train,India,224024,10388,4.637003
100,Pakistan_712873_S1Hand.tif,train,Pakistan,119557,1316,1.100730
378,Paraguay_280900_S1Hand.tif,test,Paraguay,262142,6585,2.511997
390,Paraguay_651904_S1Hand.tif,test,Paraguay,262144,27156,10.359192
30,India_285297_S1Hand.tif,train,India,245042,36006,14.693808
322,Sri-Lanka_321316_S1Hand.tif,validation,Sri-Lanka,262133,96905,36.967875
382,Paraguay_40936_S1Hand.tif,test,Paraguay,215331,0,0.000000
56,India_804466_S1Hand.tif,train,India,262134,50228,19.161192
70,Mekong_221789_S1Hand.tif,train,Mekong,215327,0,0.000000


In [16]:
# Statistics for calculated Flood extent field:

print(splits["flood_extent_percent"].describe())

count    426.000000
mean      10.655133
std       19.062733
min        0.000000
25%        0.532184
50%        2.491298
75%       11.190923
max      100.000000
Name: flood_extent_percent, dtype: float64


In [17]:
# Check for Impossible or error values:

assert (splits["flood_extent_percent"].dropna() >= 0).all()
assert (splits["flood_extent_percent"].dropna() <= 100).all()

print("✓ Flood extent values are within 0–100%.")

✓ Flood extent values are within 0–100%.


In [18]:
# Initial indication of whether the three splits have broadly comparable flood-extent distributions:

print(splits.groupby("split")["flood_extent_percent"].agg(["count", "mean", "median", "min", "max"]))

            count       mean    median  min         max
split                                                  
test           89  13.514114  2.511997  0.0  100.000000
train         251   9.378432  2.487169  0.0   97.419816
validation     86  11.422606  2.548416  0.0   92.305899


---------------------------
###### 8.1
### Section 8 — Add Image Dimensions:

In [19]:
# Function to see if every chip has same image dimensions:

def get_raster_dimensions(path):
    with rasterio.open(path) as src:
        return src.width, src.height

dimensions = splits["s1_path"].apply(get_raster_dimensions)
splits["width"] = dimensions.apply(lambda x: x[0])
splits["height"] = dimensions.apply(lambda x: x[1])

print(splits[["width", "height"]].drop_duplicates())

   width  height
0    512     512


--------------------------
###### 9.1
### Section 9 — Final Chip-level Metadata Table:

In [20]:
# Chip-Level Table:

metadata = splits[[
        "s1_id",
        "s1_filename",
        "label_filename",
        "s1_path",
        "label_path",
        "split",
        "flood_event",
        "width",
        "height",
        "valid_pixels",
        "flood_pixels",
        "flood_extent_percent"]].copy()

print("Rows:", len(metadata))
print("Columns:", len(metadata.columns))
display(metadata.sample(10))

Rows: 426
Columns: 12


,s1_id,s1_filename,label_filename,s1_path,label_path,split,flood_event,width,height,valid_pixels,flood_pixels,flood_extent_percent
422,USA_595451,USA_595451_S1Hand.tif,USA_595451_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,test,USA,512,512,262136,20531,7.832194
291,Paraguay_172476,Paraguay_172476_S1Hand.tif,Paraguay_172476_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,validation,Paraguay,512,512,262143,2815,1.073841
171,Spain_4915752,Spain_4915752_S1Hand.tif,Spain_4915752_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,train,Spain,512,512,262128,7664,2.923762
420,USA_67102,USA_67102_S1Hand.tif,USA_67102_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,test,USA,512,512,262091,9491,3.621261
157,Somalia_886726,Somalia_886726_S1Hand.tif,Somalia_886726_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,train,Somalia,512,512,262036,82995,31.673129
167,Somalia_205466,Somalia_205466_S1Hand.tif,Somalia_205466_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,train,Somalia,512,512,240083,93,0.038737
299,Paraguay_896458,Paraguay_896458_S1Hand.tif,Paraguay_896458_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,validation,Paraguay,512,512,240678,49299,20.483384
37,India_773682,India_773682_S1Hand.tif,India_773682_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,train,India,512,512,252793,30959,12.246779
184,Spain_337094,Spain_337094_S1Hand.tif,Spain_337094_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,train,Spain,512,512,262056,26075,9.950163
165,Somalia_626316,Somalia_626316_S1Hand.tif,Somalia_626316_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,train,Somalia,512,512,169085,21727,12.849750


------------------------
###### 9.2
### Sub-Section 9.2 - Create separate Bolivia holdout table:

In [21]:
# Identify the Bolivia flood event held out from the official splits.

raw_ids = {
    file.name.replace("_S1Hand.tif", "")
    for file in S1_DIR.glob("*_S1Hand.tif")
}

official_ids = set(
    pd.concat([
        train["s1_filename"],
        valid["s1_filename"],
        test["s1_filename"]
    ]).str.replace("_S1Hand.tif", "", regex=False)
)

# Identify IDs that are not part of the official splits
excluded_ids = raw_ids - official_ids

# Keep ONLY the excluded chips belonging to the Bolivia flood event.
bolivia_ids = sorted(
    image_id
    for image_id in excluded_ids
    if image_id.startswith("Bolivia_")
)

print("Official development/evaluation chips:", len(official_ids))
print("Excluded raw chips:", len(excluded_ids))
print("Bolivia holdout chips:", len(bolivia_ids))

print("\nBolivia holdout IDs:")
print(bolivia_ids)

Official development/evaluation chips: 426
Excluded raw chips: 20
Bolivia holdout chips: 15

Bolivia holdout IDs:
['Bolivia_103757', 'Bolivia_129334', 'Bolivia_195474', 'Bolivia_23014', 'Bolivia_233925', 'Bolivia_242570', 'Bolivia_290290', 'Bolivia_294583', 'Bolivia_312675', 'Bolivia_314919', 'Bolivia_360519', 'Bolivia_379434', 'Bolivia_432776', 'Bolivia_60373', 'Bolivia_76104']


In [22]:
# Restrict the holdout strictly to the Bolivia flood event:

# This prevents chips excluded from the official splits for other reasons (such as the five invalid Ghana chips) from entering the holdout.

bolivia_ids = sorted(image_id for image_id in bolivia_ids if image_id.startswith("Bolivia_"))

# Create metadata for the Bolivia holdout chips:
# The same metadata calculations used for the main dataset are applied here
# so that the holdout can be analysed consistently with the main dataset.

bolivia_records = []

for image_id in bolivia_ids:

    s1_file = S1_DIR / f"{image_id}_S1Hand.tif"
    label_file = LABEL_DIR / f"{image_id}_LabelHand.tif"

    # Read SAR image dimensions
    with rasterio.open(s1_file) as src:
        width = src.width
        height = src.height

    # Calculate label statistics
    label_stats = calculate_flood_extent(label_file)

    bolivia_records.append({
        "s1_id": image_id,
        "s1_filename": s1_file.name,
        "label_filename": label_file.name,
        "s1_path": str(s1_file),
        "label_path": str(label_file),
        "split": "bolivia_holdout",
        "flood_event": "Bolivia",
        "width": width,
        "height": height,
        "valid_pixels": label_stats["valid_pixels"],
        "flood_pixels": label_stats["flood_pixels"],
        "flood_extent_percent": label_stats["flood_extent_percent"]
    })

# Convert the records into a metadata table
bolivia_metadata = pd.DataFrame(bolivia_records)

print("Bolivia holdout rows:", len(bolivia_metadata))
display(bolivia_metadata.head())

Bolivia holdout rows: 15


,s1_id,s1_filename,label_filename,s1_path,label_path,split,flood_event,width,height,valid_pixels,flood_pixels,flood_extent_percent
0,Bolivia_103757,Bolivia_103757_S1Hand.tif,Bolivia_103757_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,88077,35362,40.148961
1,Bolivia_129334,Bolivia_129334_S1Hand.tif,Bolivia_129334_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,237814,156643,65.867863
2,Bolivia_195474,Bolivia_195474_S1Hand.tif,Bolivia_195474_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,261335,1551,0.593491
3,Bolivia_23014,Bolivia_23014_S1Hand.tif,Bolivia_23014_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,209479,7780,3.713976
4,Bolivia_233925,Bolivia_233925_S1Hand.tif,Bolivia_233925_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,150537,1,0.000664


In [23]:
print("\nBolivia flood extent statistics:")
print(bolivia_metadata["flood_extent_percent"].describe())
print("\nBolivia metadata:")
display(bolivia_metadata)


Bolivia flood extent statistics:
count    15.000000
mean     16.992148
std      21.168762
min       0.000664
25%       2.408169
50%       6.270758
75%      27.865053
max      65.867863
Name: flood_extent_percent, dtype: float64

Bolivia metadata:


,s1_id,s1_filename,label_filename,s1_path,label_path,split,flood_event,width,height,valid_pixels,flood_pixels,flood_extent_percent
0,Bolivia_103757,Bolivia_103757_S1Hand.tif,Bolivia_103757_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,88077,35362,40.148961
1,Bolivia_129334,Bolivia_129334_S1Hand.tif,Bolivia_129334_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,237814,156643,65.867863
2,Bolivia_195474,Bolivia_195474_S1Hand.tif,Bolivia_195474_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,261335,1551,0.593491
3,Bolivia_23014,Bolivia_23014_S1Hand.tif,Bolivia_23014_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,209479,7780,3.713976
4,Bolivia_233925,Bolivia_233925_S1Hand.tif,Bolivia_233925_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,150537,1,0.000664
5,Bolivia_242570,Bolivia_242570_S1Hand.tif,Bolivia_242570_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,175507,27346,15.581145
6,Bolivia_290290,Bolivia_290290_S1Hand.tif,Bolivia_290290_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,204633,19699,9.626502
7,Bolivia_294583,Bolivia_294583_S1Hand.tif,Bolivia_294583_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,130472,15008,11.502851
8,Bolivia_312675,Bolivia_312675_S1Hand.tif,Bolivia_312675_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,158008,9575,6.059820
9,Bolivia_314919,Bolivia_314919_S1Hand.tif,Bolivia_314919_LabelHand.tif,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,/Users/archiewilliamhulse/Desktop/Msc Data Sci...,bolivia_holdout,Bolivia,512,512,188845,83678,44.310413


In [24]:
# Save the Bolivia holdout metadata separately from the main dataset.

bolivia_output_file = TABLE_DIR / "bolivia_holdout_dataTable.csv"
bolivia_metadata.to_csv(bolivia_output_file, index=False)

print(f"Saved Bolivia holdout metadata to:\n{bolivia_output_file}")

Saved Bolivia holdout metadata to:
/Users/archiewilliamhulse/Desktop/Msc Data Science/Final Project/DSM500 Final Project/Results/tables/bolivia_holdout_dataTable.csv


---------------------
###### 10.1
### Section 10 — Final Checks & Save:

In [25]:
# Checks:

print("Duplicate chip IDs:", metadata["s1_id"].duplicated().sum())
print("Missing flood extent:", metadata["flood_extent_percent"].isna().sum())
print("Missing SAR files:", (~metadata["s1_path"].apply(Path.exists)).sum())
print("Missing label files:", (~metadata["label_path"].apply(Path.exists)).sum())

Duplicate chip IDs: 0
Missing flood extent: 0
Missing SAR files: 0
Missing label files: 0


In [26]:
assert metadata["s1_id"].is_unique
assert metadata["flood_extent_percent"].notna().all()
assert metadata["s1_path"].apply(Path.exists).all()
assert metadata["label_path"].apply(Path.exists).all()

print("✓ Dataset metadata passed all quality checks.")

✓ Dataset metadata passed all quality checks.


In [27]:
# Save Metadata Table to CSV:

output_file = TABLE_DIR / "chip_dataTable.csv"
metadata.to_csv(output_file, index=False)

print(f"Saved metadata to:\n{output_file}")

Saved metadata to:
/Users/archiewilliamhulse/Desktop/Msc Data Science/Final Project/DSM500 Final Project/Results/tables/chip_dataTable.csv


--------------------
###### R
### References:

Bonafilia, D., Tellman, B., Anderson, T. & Issenberg, E. (2020). Sen1Floods11: *A Georeferenced Dataset to Train and Test Deep Learning Flood Algorithms for Sentinel-1.* Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition Workshops (CVPRW), pp. 835–845.